<a href="https://colab.research.google.com/github/jmoussa99/football-ai/blob/main/Copy_of_NU_Soccer_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Football AI

---

[![GitHub](https://badges.aleen42.com/src/github.svg)](https://github.com/roboflow/sports)
[![Roboflow](https://raw.githubusercontent.com/roboflow-ai/notebooks/main/assets/badges/roboflow-blogpost.svg)](https://blog.roboflow.com/camera-calibration-sports-computer-vision/)
[![YouTube](https://badges.aleen42.com/src/youtube.svg)](https://www.youtube.com/watch?v=aBVGKoNZQUw)

Let's build a Football AI system to dig deeper into match stats! We'll use computer vision and machine learning to track players, determine which team is which, and even calculate stuff like ball possession and speed. This tutorial is perfect if you want to get hands-on with sports analytics and see how AI can take your football analysis to the next level.

![football AI diagram](https://media.roboflow.com/notebooks/examples/football-ai-diagram.png)

## Before you start

### Configure your API keys

- Open your [`HuggingFace Settings`](https://huggingface.co/settings) page. Click `Access Tokens` then `New Token` to generate new token.
- Go to your [`Roboflow Settings`](https://app.roboflow.com/settings/api) page. Click `Copy`. This will place your private key in the clipboard.
- In Colab, go to the left pane and click on `Secrets` (🔑).
    - Store HuggingFace Access Token under the name `HF_TOKEN`.
    - Store Roboflow API Key under the name `ROBOFLOW_API_KEY`.

In [ ]:
import os
from google.colab import userdata

def get_notebook_secret(name, required=False):
    value = os.environ.get(name)
    if not value:
        try:
            value = userdata.get(name)
        except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
            if required:
                raise RuntimeError(
                    f"Add {name} in Colab Secrets and enable notebook access."
                ) from exc
    if required and not value:
        raise RuntimeError(f"Set {name} in Colab Secrets or the environment.")
    if value:
        os.environ[name] = value
    return value

ROBOFLOW_API_KEY = get_notebook_secret("ROBOFLOW_API_KEY", required=True)
get_notebook_secret("HF_TOKEN")

### Select the runtime

Let's make sure that we have access to GPU. We can use `nvidia-smi` command to do that. In case of any problems navigate to `Edit` -> `Notebook settings` -> `Hardware accelerator`, set it to `GPU`, and then click `Save`.

In [ ]:
import shutil
import subprocess

HAS_NVIDIA_GPU = bool(shutil.which("nvidia-smi")) and subprocess.run(
    ["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
).returncode == 0
if HAS_NVIDIA_GPU:
    subprocess.run(["nvidia-smi"], check=True)
else:
    print("No NVIDIA GPU detected. The notebook will use CPU inference.")

## Install dependencies

Run the setup cell once, restart the Colab session, then continue at Check installed versions. This pins compatible inference, supervision, transformers, Pillow and protobuf versions for the A100 runtime.


In [ ]:
import sys
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path
from tempfile import TemporaryDirectory

# Inference 0.38 supports Python 3.9–3.12, including this notebook's Colab runtime.
if not (3, 9) <= sys.version_info[:2] < (3, 13):
    raise RuntimeError("Use a Colab Python 3.12 runtime for these dependencies.")

inference_package = "inference-gpu" if HAS_NVIDIA_GPU else "inference"
requirements = [
    f"{inference_package}==0.38.0",
    "supervision==0.25.1",
    "transformers==4.48.3",
    "pillow>=10.1,<11",
    "protobuf>=4.25.3,<6",
    "sports @ git+https://github.com/roboflow/sports.git@42c80c06b6b65a7f89455b89fe31cdf4c38ba227",
]

# Preserve Colab's CUDA build and the pandas/requests versions used by Colab itself.
constraints = []
for package in ("torch", "torchvision", "pandas", "requests"):
    try:
        constraints.append(f"{package}=={version(package)}")
    except PackageNotFoundError:
        pass

with TemporaryDirectory() as directory:
    constraint_path = Path(directory) / "constraints.txt"
    constraint_path.write_text("\n".join(constraints))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "--quiet",
        "--constraint", str(constraint_path), *requirements,
    ])

print("Installation complete. Restart the session once")
print("and continue from 'Check installed versions'; do not rerun the old install cells.")

The sports package is installed in the setup cell above.


In [ ]:
# sports is installed with the other pinned dependencies in the setup cell above.


Check the installed versions after restarting the session.


In [ ]:
from importlib.metadata import PackageNotFoundError, version

for package in ("inference-gpu", "inference", "sports", "supervision", "transformers", "protobuf", "pillow", "torch"):
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        pass  # Only one of inference-gpu or inference is installed.


In [ ]:
import os
import torch
import onnxruntime as ort

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
available_providers = ort.get_available_providers()
providers = ["CPUExecutionProvider"]
if DEVICE == "cuda" and "CUDAExecutionProvider" in available_providers:
    providers.insert(0, "CUDAExecutionProvider")

# Set these before importing inference. Keep CPU fallback for detection models.
os.environ["USE_TF"] = "0"  # The notebook uses PyTorch; avoid optional TensorFlow imports.
os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[" + ",".join(providers) + "]"
for model in ("CLIP", "SAM", "SAM2", "OWLV2", "GAZE", "DOCTR", "TROCR",
              "GROUNDINGDINO", "YOLO_WORLD"):
    os.environ[f"CORE_MODEL_{model}_ENABLED"] = "False"

print(f"Team classifier device: {DEVICE}")
print(f"Detection execution providers: {providers}")
if DEVICE == "cpu":
    print("CUDA is unavailable in PyTorch; continuing on CPU.")

## ball, player, goalkeeper and referee detection

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import numpy as np
import supervision as sv

# Change this one path if your Drive video has a different filename.
SOURCE_VIDEO_PATH = "/content/drive/MyDrive/Soccer_Video_Analysis/04-18-26-uic-vs-northwestern-aj.mp4"
if not Path(SOURCE_VIDEO_PATH).is_file():
    raise FileNotFoundError(
        f"Video not found: {SOURCE_VIDEO_PATH}. Mount Drive and check the path.")

video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
if not np.isfinite(video_info.fps) or video_info.fps <= 0:
    raise ValueError("The input video must have a valid, positive frame rate.")
if video_info.total_frames <= 0:
    raise ValueError("The input video contains no readable frames.")
OUTPUT_VIDEO_PATH = str(Path(SOURCE_VIDEO_PATH).with_name(
    Path(SOURCE_VIDEO_PATH).stem + "_output_tracking.mp4"))
print(video_info)
frame = next(sv.get_video_frames_generator(SOURCE_VIDEO_PATH))
sv.plot_image(frame)


In [ ]:
# Pillow is installed at a compatible version in the setup cell above.


In [ ]:
# Reload secrets here too, so this cell works after a session restart.
import os
from google.colab import userdata
from inference import get_model

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    try:
        ROBOFLOW_API_KEY = userdata.get("ROBOFLOW_API_KEY")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as exc:
        raise RuntimeError(
            "Add ROBOFLOW_API_KEY in Colab Secrets and enable notebook access."
        ) from exc
if not ROBOFLOW_API_KEY:
    raise RuntimeError("ROBOFLOW_API_KEY is required to load the detection models.")
os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

if not os.environ.get("HF_TOKEN"):
    try:
        hf_token = userdata.get("HF_TOKEN")
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        hf_token = None
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token

PLAYER_DETECTION_MODEL_ID = "football-players-detection-3zvbc/11"
FIELD_DETECTION_MODEL_ID = "football-field-detection-f07vi/14"
PLAYER_DETECTION_MODEL = get_model(
    model_id=PLAYER_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)
FIELD_DETECTION_MODEL = get_model(
    model_id=FIELD_DETECTION_MODEL_ID, api_key=ROBOFLOW_API_KEY)

In [ ]:
import supervision as sv


box_annotator = sv.BoxAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#FF8C00', '#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000')
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
detections = sv.Detections.from_inference(result)

labels = [
    f"{class_name} {confidence:.2f}"
    for class_name, confidence
    in zip(detections['class_name'], detections.confidence)
]

annotated_frame = frame.copy()
annotated_frame = box_annotator.annotate(
    scene=annotated_frame,
    detections=detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=detections,
    labels=labels)

sv.plot_image(annotated_frame)

## video game style visualization

In [ ]:
import supervision as sv

BALL_ID = 0

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
detections = sv.Detections.from_inference(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections.class_id -= 1

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

## player tracking

In [ ]:
import supervision as sv

BALL_ID = 0

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
detections = sv.Detections.from_inference(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections.class_id -= 1
all_detections = tracker.update_with_detections(detections=all_detections)

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

## split players into teams

![football AI diagram](https://media.roboflow.com/notebooks/examples/football-ai-team-clustering.png)

**Note:** Before training our player clustering model, we need to gather training data. To do this, we'll sample one frame per second, detect players within those frames, and then crop them out.

In [ ]:
import numpy as np
import supervision as sv
from tqdm.auto import tqdm
from sports.common.team import TeamClassifier

BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3
STRIDE = 30  # Sample every STRIDE frames for classifier training.

def crop_players(frame, detections):
    """Return aligned detections/crops, skipping empty or invalid image regions."""
    height, width = frame.shape[:2]
    indices, crops = [], []
    for index, box in enumerate(detections.xyxy):
        if not np.isfinite(box).all():
            continue
        x1, y1, x2, y2 = np.clip(box, [0, 0, 0, 0], [width, height, width, height]).astype(int)
        if x2 <= x1 or y2 <= y1:
            continue
        indices.append(index)
        crops.append(frame[y1:y2, x1:x2])
    return detections[np.asarray(indices, dtype=int)], crops

crops = []
frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)
for frame in tqdm(frame_generator, desc="Collecting player crops"):
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
    detections = sv.Detections.from_inference(result)
    players = detections[detections.class_id == PLAYER_ID].with_nms(threshold=0.5)
    _, players_crops = crop_players(frame, players)
    crops.extend(players_crops)

# Three-dimensional UMAP needs more than four samples for spectral initialization.
if len(crops) < 6:
    raise ValueError(
        f"Only {len(crops)} valid player crops found. Use a longer video or lower STRIDE.")
print(f"Collected {len(crops)} player crops.")

**Note:** Here's a sample (100 elements) of the crops we've gathered.

In [ ]:
if crops:
    preview = crops[:100]
    columns = min(10, len(preview))
    rows = (len(preview) + columns - 1) // columns
    sv.plot_images_grid(preview, grid_size=(rows, columns))


**Note:** Next, we'll run [SigLIP](https://huggingface.co/docs/transformers/en/model_doc/siglip) to calculate embeddings for each of the crops.

In [ ]:
import torch
from transformers import AutoProcessor, SiglipVisionModel

SIGLIP_MODEL_PATH = 'google/siglip-base-patch16-224'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
EMBEDDINGS_MODEL = SiglipVisionModel.from_pretrained(SIGLIP_MODEL_PATH).to(DEVICE)
EMBEDDINGS_PROCESSOR = AutoProcessor.from_pretrained(SIGLIP_MODEL_PATH)

In [ ]:
import numpy as np
from more_itertools import chunked

BATCH_SIZE = 32

pil_crops = [sv.cv2_to_pillow(crop) for crop in crops]
batches = chunked(pil_crops, BATCH_SIZE)
data = []
with torch.no_grad():
    for batch in tqdm(batches, desc='embedding extraction'):
        inputs = EMBEDDINGS_PROCESSOR(images=batch, return_tensors="pt").to(DEVICE)
        outputs = EMBEDDINGS_MODEL(**inputs)
        embeddings = torch.mean(outputs.last_hidden_state, dim=1).cpu().numpy()
        data.append(embeddings)

data = np.concatenate(data)

**Note:** Using [UMAP](https://github.com/lmcinnes/umap), we project our embeddings from `(N, 768)` to `(N, 3)` and then perform a two-cluster division using [KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html).

In [ ]:
import sys
from unittest.mock import patch
from sklearn.cluster import KMeans

# UMAP's optional ParametricUMAP imports Colab's TensorFlow even though this
# notebook only uses regular UMAP. Skip that optional import, which can call
# a removed protobuf MessageFactory method on runtime 2026.04.
with patch.dict(sys.modules, {"tensorflow": None}):
    import umap

REDUCER = umap.UMAP(n_components=3, random_state=42)
CLUSTERING_MODEL = KMeans(n_clusters=2, random_state=42, n_init=10)


In [ ]:
if len(data) < 6:
    raise ValueError("UMAP needs at least 6 player crops; use a longer video or lower STRIDE.")
projections = REDUCER.fit_transform(data)
clusters = CLUSTERING_MODEL.fit_predict(projections)


**Note:** Here's an interactive visualization of our results. Click on a dot to display its associated crop.

In [ ]:
import plotly.graph_objects as go
import numpy as np
from typing import Dict, List
from IPython.display import display, HTML
from PIL import Image
import base64
from io import BytesIO


def pil_image_to_data_uri(image: Image.Image) -> str:
    buffered = BytesIO()
    image.save(buffered, format="PNG")
    img_str = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{img_str}"


def display_projections(
    labels: np.ndarray,
    projections: np.ndarray,
    images: List[Image.Image],
    show_legend: bool = False,
    show_markers_with_text: bool = True
) -> None:
    image_data_uris = {f"image_{i}": pil_image_to_data_uri(image) for i, image in enumerate(images)}
    image_ids = np.array([f"image_{i}" for i in range(len(images))])

    unique_labels = np.unique(labels)
    traces = []
    for unique_label in unique_labels:
        mask = labels == unique_label
        try:
          customdata_masked = image_ids[mask]

          trace = go.Scatter3d(
            x=projections[mask][:, 0],
            y=projections[mask][:, 1],
            z=projections[mask][:, 2],
            mode='markers+text' if show_markers_with_text else 'markers',
            text=labels[mask],
            customdata=customdata_masked,
            name=str(unique_label),
            marker=dict(size=8),
            hovertemplate="<b>class: %{text}</b><br>image ID: %{customdata}<extra></extra>"
          )
          traces.append(trace)

        except IndexError:
          pass

    # Calculate shared range for cube appearance
    all_axes = projections
    min_val = np.min(all_axes)
    max_val = np.max(all_axes)
    padding = (max_val - min_val) * 0.05
    axis_range = [min_val - padding, max_val + padding]

    fig = go.Figure(data=traces)
    fig.update_layout(
        scene=dict(
            xaxis=dict(title='X', range=axis_range),
            yaxis=dict(title='Y', range=axis_range),
            zaxis=dict(title='Z', range=axis_range),
            aspectmode='cube'  # Ensures equal scaling
        ),
        width=1000,
        height=1000,
        showlegend=show_legend
    )

    plotly_div = fig.to_html(full_html=False, include_plotlyjs=False, div_id="scatter-plot-3d")

    javascript_code = f"""
    <script>
        function displayImage(imageId) {{
            var imageElement = document.getElementById('image-display');
            var placeholderText = document.getElementById('placeholder-text');
            var imageDataURIs = {image_data_uris};
            imageElement.src = imageDataURIs[imageId];
            imageElement.style.display = 'block';
            placeholderText.style.display = 'none';
        }}

        var chartElement = document.getElementById('scatter-plot-3d');

        chartElement.on('plotly_click', function(data) {{
            var customdata = data.points[0].customdata;
            displayImage(customdata);
        }});
    </script>
    """

    html_template = f"""
    <!DOCTYPE html>
    <html>
        <head>
            <script src="https://cdn.plot.ly/plotly-latest.min.js"></script>
            <style>
                #image-container {{
                    position: fixed;
                    top: 0;
                    left: 0;
                    width: 200px;
                    height: 200px;
                    padding: 5px;
                    border: 1px solid #ccc;
                    background-color: white;
                    z-index: 1000;
                    box-sizing: border-box;
                    display: flex;
                    align-items: center;
                    justify-content: center;
                    text-align: center;
                }}
                #image-display {{
                    width: 100%;
                    height: 100%;
                    object-fit: contain;
                }}
            </style>
        </head>
        <body>
            {plotly_div}
            <div id="image-container">
                <img id="image-display" src="" alt="Selected image" style="display: none;" />
                <p id="placeholder-text">Click on a data entry to display an image</p>
            </div>
            {javascript_code}
        </body>
    </html>
    """

    display(HTML(html_template))

display_projections(clusters, projections, pil_crops)

**Note:** To simplify the use of the SigLIP, UMAP, and KMeans combo, I've packaged all these models into a [`TeamClassifier`](https://github.com/roboflow/sports/blob/06053616f1f8a8ae1fa936eb00dcdc2e4f888bb1/sports/common/team.py#L41) that you can find in the [sports](https://github.com/roboflow/sports) repository.

In [ ]:
from sports.common.team import TeamClassifier

# Reuse the player-only BGR crops collected above. The embedding preview uses
# separate PIL copies, so the classifier receives the original image format.
if len(crops) < 6:
    raise ValueError("At least 6 valid player crops are needed to train the team classifier.")
if "EMBEDDINGS_MODEL" in globals():
    del EMBEDDINGS_MODEL
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
team_classifier = TeamClassifier(device=DEVICE)
team_classifier.fit(crops)


**Note:** Time to assign goalkeepers to teams. We'll use a simple heuristic: calculate the average position (centroid) of the players belonging to both teams and then assign the goalkeeper to the team whose average position is closer.

In [ ]:
import cv2
import numpy as np
import supervision as sv

UNKNOWN_TEAM_ID = 3  # Neutral gray; referee annotations use class 2.

def resolve_goalkeepers_team_id(players, goalkeepers):
    """Assign by nearest team centroid when both teams are visible."""
    team_ids = np.full(len(goalkeepers), UNKNOWN_TEAM_ID, dtype=int)
    if not len(goalkeepers) or not len(players):
        return team_ids
    players_xy = players.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    team_0 = players_xy[players.class_id == 0]
    team_1 = players_xy[players.class_id == 1]
    if not len(team_0) or not len(team_1):
        return team_ids
    centroids = np.array([team_0.mean(axis=0), team_1.mean(axis=0)])
    goalkeepers_xy = goalkeepers.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    distances = np.linalg.norm(goalkeepers_xy[:, None, :] - centroids[None, :, :], axis=2)
    return np.argmin(distances, axis=1).astype(int)
from sports.common.view import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration

CONFIG = SoccerPitchConfiguration()

def get_pitch_transformer(key_points, config=CONFIG, target_scale=1.0):
    """Return None for missing/insufficient keypoints or a degenerate homography."""
    if key_points is None or key_points.confidence is None or len(key_points.xy) == 0:
        return None
    xy = key_points.xy[0]
    if len(xy) == 0:
        return None
    confidence = key_points.confidence[0]
    vertices = np.asarray(config.vertices, dtype=np.float32)
    if len(xy) != len(vertices) or len(confidence) != len(xy):
        raise ValueError("Field-model keypoints do not match the configured pitch vertices.")
    valid = (confidence > 0.5) & np.isfinite(confidence) & np.isfinite(xy).all(axis=1)
    if np.count_nonzero(valid) < 4:
        return None
    source = xy[valid]
    target = vertices[valid] * target_scale
    if (np.linalg.matrix_rank(source - source.mean(axis=0)) < 2
            or np.linalg.matrix_rank(target - target.mean(axis=0)) < 2):
        return None
    try:
        transformer = ViewTransformer(source=source, target=target)
    except (ValueError, cv2.error):
        return None
    if not np.isfinite(transformer.m).all() or np.linalg.matrix_rank(transformer.m) < 3:
        return None
    return transformer



In [ ]:
import supervision as sv

BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=25,
    height=21,
    outline_thickness=1
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
detections = sv.Detections.from_inference(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections = tracker.update_with_detections(detections=all_detections)

goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
players_detections = all_detections[all_detections.class_id == PLAYER_ID]
referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

players_detections, players_crops = crop_players(frame, players_detections)
players_detections.class_id = (np.asarray(team_classifier.predict(players_crops), dtype=int)
                               if players_crops else np.empty(0, dtype=int))

goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
    players_detections, goalkeepers_detections)

referees_detections.class_id = np.full(len(referees_detections), 2, dtype=int)

all_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections, referees_detections])

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

all_detections.class_id = all_detections.class_id.astype(int)

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

## pitch keypoint detection

In [ ]:
# FIELD_DETECTION_MODEL was loaded with PLAYER_DETECTION_MODEL above.


In [ ]:
import supervision as sv


vertex_annotator = sv.VertexAnnotator(
    color=sv.Color.from_hex('#FF1493'),
    radius=8)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
key_points = sv.KeyPoints.from_inference(result)

annotated_frame = frame.copy()
annotated_frame = vertex_annotator.annotate(
    scene=annotated_frame,
    key_points=key_points)

sv.plot_image(annotated_frame)

**Note:** Notice that some of the keypoints we detected are in incorrect locations. These are keypoints with a low confidence level. Let's filter out these keypoints and keep only the ones the model is confident about.

## filter low confidence keypoints

In [ ]:
import supervision as sv


vertex_annotator = sv.VertexAnnotator(
    color=sv.Color.from_hex('#FF1493'),
    radius=8)

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH, start=min(200, video_info.total_frames - 1))
frame = next(frame_generator)

result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
key_points = sv.KeyPoints.from_inference(result)
if key_points is not None and key_points.confidence is not None and len(key_points.xy):
  valid = key_points.confidence[0] > 0.5
  frame_reference_points = key_points.xy[0][valid]
  frame_reference_key_points = sv.KeyPoints(
      xy=frame_reference_points[np.newaxis, ...])

  annotated_frame = frame.copy()
  annotated_frame = vertex_annotator.annotate(
    scene=annotated_frame,
    key_points=frame_reference_key_points)
else:
  annotated_frame = frame.copy()
sv.plot_image(annotated_frame)

## project pitch lines on frame

**Note:** The [sports](https://github.com/roboflow/sports) repository contains a [`SoccerPitchConfiguration`](https://github.com/roboflow/sports/blob/06053616f1f8a8ae1fa936eb00dcdc2e4f888bb1/sports/configs/soccer.py#L6) that provides information about the real-world geometry of the soccer pitch. It also includes utilities for visualizing elements located on the pitch.

In [ ]:
from sports.annotators.soccer import draw_pitch
from sports.configs.soccer import SoccerPitchConfiguration

CONFIG = SoccerPitchConfiguration()

annotated_frame = draw_pitch(CONFIG)

sv.plot_image(annotated_frame)

**Note:** It's time to utilize the keypoint pairs located on the camera perspective plane and the football pitch plane. The [sports](https://github.com/roboflow/sports) repository includes a [`ViewTransformer`](https://github.com/roboflow/sports/blob/06053616f1f8a8ae1fa936eb00dcdc2e4f888bb1/sports/common/view.py#L7), which employs homography for perspective transformation.

In [ ]:
import numpy as np
import supervision as sv
from sports.common.view import ViewTransformer
from sports.configs.soccer import SoccerPitchConfiguration

edge_annotator = sv.EdgeAnnotator(
    color=sv.Color.from_hex('#00BFFF'), thickness=2, edges=CONFIG.edges)
vertex_annotator = sv.VertexAnnotator(color=sv.Color.from_hex('#FF1493'), radius=8)
vertex_annotator_2 = sv.VertexAnnotator(color=sv.Color.from_hex('#00BFFF'), radius=8)

frame = next(sv.get_video_frames_generator(
    SOURCE_VIDEO_PATH, start=min(200, video_info.total_frames - 1)))
result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
key_points = sv.KeyPoints.from_inference(result)
annotated_frame = frame.copy()
if key_points is not None and key_points.confidence is not None and len(key_points.xy):
    xy = key_points.xy[0]
    vertices = np.asarray(CONFIG.vertices, dtype=np.float32)
    if len(xy) != len(vertices):
        raise ValueError("Field-model keypoints do not match the configured pitch vertices.")
    valid = (key_points.confidence[0] > 0.5) & np.isfinite(xy).all(axis=1)
    if valid.sum() >= 4:
        pitch_points, frame_points = vertices[valid], xy[valid]
        if np.linalg.matrix_rank(frame_points - frame_points.mean(axis=0)) == 2:
            try:
                transformer = ViewTransformer(source=pitch_points, target=frame_points)
            except (ValueError, cv2.error):
                transformer = None
            if transformer is not None:
                frame_all_points = transformer.transform_points(vertices)
                all_key_points = sv.KeyPoints(xy=frame_all_points[np.newaxis, ...])
                reference_points = sv.KeyPoints(xy=frame_points[np.newaxis, ...])
                annotated_frame = edge_annotator.annotate(annotated_frame, all_key_points)
                annotated_frame = vertex_annotator_2.annotate(annotated_frame, all_key_points)
                annotated_frame = vertex_annotator.annotate(annotated_frame, reference_points)
            else:
                print("Pitch projection unavailable for this frame.")
    else:
        print("Fewer than four confident pitch keypoints; showing the original frame.")
sv.plot_image(annotated_frame)


## project ball, players and referies on pitch

In [ ]:
import supervision as sv
from tqdm import tqdm
from sports.common.team import TeamClassifier

PLAYER_ID = 2
STRIDE = 30

frame_generator = sv.get_video_frames_generator(
    source_path=SOURCE_VIDEO_PATH, stride=STRIDE)

crops = []
for frame in tqdm(frame_generator, desc='collecting crops'):
    result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
    detections = sv.Detections.from_inference(result)
    players_detections = detections[detections.class_id == PLAYER_ID]
    players_crops = [sv.crop_image(frame, xyxy) for xyxy in detections.xyxy]
    crops += players_crops

team_classifier = TeamClassifier(device="cuda")
team_classifier.fit(crops)

In [ ]:
import cv2
import numpy as np
import supervision as sv
from scipy.spatial.distance import cdist
from typing import Optional

def draw_pitch_voronoi_diagram_2(
    config: SoccerPitchConfiguration,
    team_1_xy: np.ndarray,
    team_2_xy: np.ndarray,
    team_1_color: sv.Color = sv.Color.RED,
    team_2_color: sv.Color = sv.Color.WHITE,
    opacity: float = 0.5,
    padding: int = 50,
    scale: float = 0.1,
    pitch: Optional[np.ndarray] = None
) -> np.ndarray:
    """Blend the two teams' nearest-player distances over the pitch."""
    if pitch is None:
        pitch = draw_pitch(config=config, padding=padding, scale=scale)
    team_1_xy = np.asarray(team_1_xy, dtype=np.float32).reshape(-1, 2)
    team_2_xy = np.asarray(team_2_xy, dtype=np.float32).reshape(-1, 2)
    team_1_xy = team_1_xy[np.isfinite(team_1_xy).all(axis=1)]
    team_2_xy = team_2_xy[np.isfinite(team_2_xy).all(axis=1)]
    if not len(team_1_xy) or not len(team_2_xy):
        return pitch

    height, width = pitch.shape[:2]
    voronoi = np.empty_like(pitch)
    color_1 = np.asarray(team_1_color.as_bgr(), dtype=np.float32)
    color_2 = np.asarray(team_2_color.as_bgr(), dtype=np.float32)
    for row in range(0, height, 64):
        stop = min(row + 64, height)
        y, x = np.mgrid[row:stop, 0:width]
        grid = np.column_stack(((x - padding).ravel(), (y - padding).ravel()))
        distance_1 = cdist(grid, team_1_xy * scale).min(axis=1)
        distance_2 = cdist(grid, team_2_xy * scale).min(axis=1)
        ratio = distance_2 / np.maximum(distance_1 + distance_2, 1e-5)
        blend = (np.tanh((ratio - 0.5) * 15) * 0.5 + 0.5)[:, None]
        voronoi[row:stop] = (blend * color_1 + (1 - blend) * color_2).reshape(stop-row, width, 3).astype(np.uint8)
    return cv2.addWeighted(voronoi, opacity, pitch, 1 - opacity, 0)


In [ ]:
import supervision as sv
from sports.annotators.soccer import (
    draw_pitch,
    draw_points_on_pitch,
    draw_pitch_voronoi_diagram
)

BALL_ID = 0
GOALKEEPER_ID = 1
PLAYER_ID = 2
REFEREE_ID = 3

ellipse_annotator = sv.EllipseAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080']),
    thickness=2
)
label_annotator = sv.LabelAnnotator(
    color=sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080']),
    text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER
)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'),
    base=20, height=17
)

tracker = sv.ByteTrack()
tracker.reset()

frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(frame_generator)

# ball, goalkeeper, player, referee detection

result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
detections = sv.Detections.from_inference(result)

ball_detections = detections[detections.class_id == BALL_ID]
ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

all_detections = detections[detections.class_id != BALL_ID]
all_detections = all_detections.with_nms(threshold=0.5, class_agnostic=True)
all_detections = tracker.update_with_detections(detections=all_detections)

goalkeepers_detections = all_detections[all_detections.class_id == GOALKEEPER_ID]
players_detections = all_detections[all_detections.class_id == PLAYER_ID]
referees_detections = all_detections[all_detections.class_id == REFEREE_ID]

# team assignment

players_detections, players_crops = crop_players(frame, players_detections)
players_detections.class_id = (np.asarray(team_classifier.predict(players_crops), dtype=int)
                               if players_crops else np.empty(0, dtype=int))

goalkeepers_detections.class_id = resolve_goalkeepers_team_id(
    players_detections, goalkeepers_detections)

referees_detections.class_id = np.full(len(referees_detections), 2, dtype=int)

all_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections, referees_detections])

# frame visualization

labels = [
    f"#{tracker_id}"
    for tracker_id
    in all_detections.tracker_id
]

all_detections.class_id = all_detections.class_id.astype(int)

annotated_frame = frame.copy()
annotated_frame = ellipse_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections)
annotated_frame = label_annotator.annotate(
    scene=annotated_frame,
    detections=all_detections,
    labels=labels)
annotated_frame = triangle_annotator.annotate(
    scene=annotated_frame,
    detections=ball_detections)

sv.plot_image(annotated_frame)

players_detections = sv.Detections.merge([
    players_detections, goalkeepers_detections
])

# detect pitch key points

result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
key_points = sv.KeyPoints.from_inference(result)

# project ball, players and referies on pitch

transformer = get_pitch_transformer(key_points)
if transformer is None:
    print("Pitch projection unavailable for this frame; showing detections only.")
else:
    frame_ball_xy = ball_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_ball_xy = transformer.transform_points(frame_ball_xy)
    players_xy = players_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_players_xy = transformer.transform_points(players_xy)
    referees_xy = referees_detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    pitch_referees_xy = transformer.transform_points(referees_xy)

    # visualize video game-style radar view

    annotated_frame = draw_pitch(CONFIG)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_ball_xy,
        face_color=sv.Color.WHITE,
        edge_color=sv.Color.BLACK,
        radius=10,
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_players_xy[players_detections.class_id == 0],
        face_color=sv.Color.from_hex('00BFFF'),
        edge_color=sv.Color.BLACK,
        radius=16,
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_players_xy[players_detections.class_id == 1],
        face_color=sv.Color.from_hex('FF1493'),
        edge_color=sv.Color.BLACK,
        radius=16,
        pitch=annotated_frame)
    annotated_frame = draw_points_on_pitch(
        config=CONFIG,
        xy=pitch_referees_xy,
        face_color=sv.Color.from_hex('FFD700'),
        edge_color=sv.Color.BLACK,
        radius=16,
        pitch=annotated_frame)

    sv.plot_image(annotated_frame)

    if np.any(players_detections.class_id == 0) and np.any(players_detections.class_id == 1):
        # visualize voronoi diagram

        annotated_frame = draw_pitch(CONFIG)
        annotated_frame = draw_pitch_voronoi_diagram(
            config=CONFIG,
            team_1_xy=pitch_players_xy[players_detections.class_id == 0],
            team_2_xy=pitch_players_xy[players_detections.class_id == 1],
            team_1_color=sv.Color.from_hex('00BFFF'),
            team_2_color=sv.Color.from_hex('FF1493'),
            pitch=annotated_frame)

        sv.plot_image(annotated_frame)

        # visualize voronoi diagram with blend

        annotated_frame = draw_pitch(
            config=CONFIG,
            background_color=sv.Color.WHITE,
            line_color=sv.Color.BLACK
        )
        annotated_frame = draw_pitch_voronoi_diagram_2(
            config=CONFIG,
            team_1_xy=pitch_players_xy[players_detections.class_id == 0],
            team_2_xy=pitch_players_xy[players_detections.class_id == 1],
            team_1_color=sv.Color.from_hex('00BFFF'),
            team_2_color=sv.Color.from_hex('FF1493'),
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_ball_xy,
            face_color=sv.Color.WHITE,
            edge_color=sv.Color.WHITE,
            radius=8,
            thickness=1,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_players_xy[players_detections.class_id == 0],
            face_color=sv.Color.from_hex('00BFFF'),
            edge_color=sv.Color.WHITE,
            radius=16,
            thickness=1,
            pitch=annotated_frame)
        annotated_frame = draw_points_on_pitch(
            config=CONFIG,
            xy=pitch_players_xy[players_detections.class_id == 1],
            face_color=sv.Color.from_hex('FF1493'),
            edge_color=sv.Color.WHITE,
            radius=16,
            thickness=1,
            pitch=annotated_frame)

        sv.plot_image(annotated_frame)


## ball tracking

In [ ]:
from collections import deque
import supervision as sv
from sports.annotators.soccer import draw_pitch, draw_points_on_pitch

BALL_ID = 0
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)
path_raw = []
for frame in tqdm(sv.get_video_frames_generator(SOURCE_VIDEO_PATH), total=video_info.total_frames):
    detections = sv.Detections.from_inference(
        PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0])
    balls = detections[detections.class_id == BALL_ID]
    key_points = sv.KeyPoints.from_inference(
        FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0])
    transformer = get_pitch_transformer(key_points)
    if transformer is None or len(balls) != 1:
        path_raw.append(np.empty((0, 2), dtype=np.float32))
        continue
    ball_xy = balls.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
    path_raw.append(transformer.transform_points(ball_xy))


In [ ]:
path = [
    points.reshape(2) if points.shape == (1, 2) else np.empty(0, dtype=np.float32)
    for points in path_raw
]


In [ ]:
from sports.annotators.soccer import draw_paths_on_pitch

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_paths_on_pitch(
    config=CONFIG,
    paths=[path],
    color=sv.Color.WHITE,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

In [ ]:
from typing import List

fps = video_info.fps
speeds = []  # One value per frame, in m/s; None means no reliable measurement.

def replace_outliers_based_on_distance(
    positions: List[np.ndarray], distance_threshold: float
) -> List[np.ndarray]:
    last_valid_position = None
    last_valid_index = None
    cleaned_positions = []
    speeds.clear()
    for index, position in enumerate(positions):
        if len(position) != 2 or not np.isfinite(position).all():
            cleaned_positions.append(np.empty(0, dtype=np.float32))
            speeds.append(None)
            continue
        position = np.asarray(position, dtype=np.float32)
        if last_valid_position is None:
            cleaned_positions.append(position)
            speeds.append(None)
            last_valid_position, last_valid_index = position, index
            continue
        distance_cm = float(np.linalg.norm(position - last_valid_position))
        if distance_cm > distance_threshold:
            cleaned_positions.append(np.empty(0, dtype=np.float32))
            speeds.append(None)
            continue
        elapsed_seconds = (index - last_valid_index) / fps
        speeds.append(distance_cm / 100.0 / elapsed_seconds)
        cleaned_positions.append(position)
        last_valid_position, last_valid_index = position, index
    return cleaned_positions


In [ ]:
MAX_DISTANCE_THRESHOLD = 5000

path = replace_outliers_based_on_distance(path, MAX_DISTANCE_THRESHOLD)

In [ ]:
from sports.annotators.soccer import draw_paths_on_pitch

annotated_frame = draw_pitch(CONFIG)
annotated_frame = draw_paths_on_pitch(
    config=CONFIG,
    paths=[path],
    color=sv.Color.WHITE,
    pitch=annotated_frame)

sv.plot_image(annotated_frame)

In [ ]:
import cv2
import numpy as np
from scipy.spatial.distance import cdist

def draw_all_connections_on_frame(
    frame_image: np.ndarray,
    detections: sv.Detections,
    color: sv.Color,
    line_thickness: int = 1,
    num_neighbors: int = 2,
) -> np.ndarray:
    if len(detections) < 2 or num_neighbors <= 0:
        return frame_image
    output_image = frame_image.copy()
    points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER).astype(int)
    distances = cdist(points, points)
    np.fill_diagonal(distances, np.inf)
    drawn_connections = set()
    count = min(num_neighbors, len(points) - 1)
    for index, neighbors in enumerate(np.argsort(distances, axis=1)[:, :count]):
        for neighbor in neighbors:
            connection = tuple(sorted((index, int(neighbor))))
            if connection not in drawn_connections:
                cv2.line(output_image, tuple(points[index]), tuple(points[neighbor]),
                         color.as_bgr(), line_thickness, cv2.LINE_AA)
                drawn_connections.add(connection)
    return output_image
class PlayerSpeedEstimator:
    """Measure displacement between sampled frames in pixels and pitch meters."""
    def __init__(self, fps, sample_stride=30):
        if not np.isfinite(fps) or fps <= 0 or sample_stride < 1:
            raise ValueError("fps must be positive and sample_stride must be at least 1.")
        self.fps = fps
        self.sample_stride = sample_stride
        self.last_sample_frame = None
        self.pixel_positions = {}
        self.meter_positions = {}
        self.speeds_pps = {}
        self.speeds_mps = {}

    def update(self, frame_index, detections, transformer=None):
        tracker_ids = detections.tracker_id
        if tracker_ids is None:
            tracker_ids = np.empty(0, dtype=int)
        visible_ids = set(tracker_ids)
        # A missed detection breaks the measurement interval for that track.
        for history in (self.pixel_positions, self.meter_positions,
                        self.speeds_pps, self.speeds_mps):
            for tracker_id in list(history):
                if tracker_id not in visible_ids:
                    del history[tracker_id]
        for tracker_id in tracker_ids:
            self.speeds_pps.setdefault(tracker_id, 0.0)
            self.speeds_mps.setdefault(tracker_id, None)
        if frame_index % self.sample_stride:
            return self.speeds_pps, self.speeds_mps

        points = detections.get_anchors_coordinates(sv.Position.BOTTOM_CENTER)
        pixels = {tracker_id: point.copy() for tracker_id, point in zip(tracker_ids, points)}
        meters = {}
        if transformer is not None and len(points):
            transformed = transformer.transform_points(points)
            meters = {tracker_id: point.copy() for tracker_id, point in zip(tracker_ids, transformed)
                      if np.isfinite(point).all()}

        elapsed = None if self.last_sample_frame is None else (frame_index - self.last_sample_frame) / self.fps
        for tracker_id in tracker_ids:
            self.speeds_pps[tracker_id] = 0.0
            self.speeds_mps[tracker_id] = None
            if elapsed is not None and elapsed > 0:
                if tracker_id in pixels and tracker_id in self.pixel_positions:
                    self.speeds_pps[tracker_id] = float(np.linalg.norm(
                        pixels[tracker_id] - self.pixel_positions[tracker_id]) / elapsed)
                if tracker_id in meters and tracker_id in self.meter_positions:
                    self.speeds_mps[tracker_id] = float(np.linalg.norm(
                        meters[tracker_id] - self.meter_positions[tracker_id]) / elapsed)

        self.pixel_positions = pixels
        self.meter_positions = meters
        self.last_sample_frame = frame_index
        return self.speeds_pps, self.speeds_mps
SPEED_STRIDE = 30
DRAW_TEAM_CONNECTIONS = False

palette = sv.ColorPalette.from_hex(['#00BFFF', '#FF1493', '#FFD700', '#808080'])
ellipse_annotator = sv.EllipseAnnotator(color=palette, thickness=2)
triangle_annotator = sv.TriangleAnnotator(
    color=sv.Color.from_hex('#FFD700'), base=25, height=21, outline_thickness=1)
label_annotator = sv.LabelAnnotator(
    color=palette, text_color=sv.Color.from_hex('#000000'),
    text_position=sv.Position.BOTTOM_CENTER)

tracker = sv.ByteTrack(frame_rate=int(round(video_info.fps)))
speed_estimator = PlayerSpeedEstimator(video_info.fps, sample_stride=SPEED_STRIDE)
frames_without_calibration = 0
processed_frames = 0

with sv.VideoSink(OUTPUT_VIDEO_PATH, video_info) as sink:
    frames = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
    for frame_index, frame in tqdm(enumerate(frames), total=video_info.total_frames,
                                   desc="Tracking video"):
        result = PLAYER_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
        detections = sv.Detections.from_inference(result)
        ball_detections = detections[detections.class_id == BALL_ID]
        ball_detections.xyxy = sv.pad_boxes(xyxy=ball_detections.xyxy, px=10)

        people = detections[detections.class_id != BALL_ID].with_nms(
            threshold=0.5, class_agnostic=True)
        people = tracker.update_with_detections(detections=people)
        goalkeepers = people[people.class_id == GOALKEEPER_ID]
        players = people[people.class_id == PLAYER_ID]
        referees = people[people.class_id == REFEREE_ID]

        players, player_crops = crop_players(frame, players)
        players.class_id = (np.asarray(team_classifier.predict(player_crops), dtype=int)
                            if player_crops else np.empty(0, dtype=int))
        goalkeepers.class_id = resolve_goalkeepers_team_id(players, goalkeepers)
        referees.class_id = np.full(len(referees), 2, dtype=int)

        participants = sv.Detections.merge([players, goalkeepers])
        final_detections = sv.Detections.merge([participants, referees])

        transformer = None
        if frame_index % SPEED_STRIDE == 0:
            field_result = FIELD_DETECTION_MODEL.infer(frame, confidence=0.3)[0]
            transformer = get_pitch_transformer(sv.KeyPoints.from_inference(field_result), target_scale=0.01)
            if transformer is None:
                frames_without_calibration += 1
        player_speeds_pps, player_speeds_mps = speed_estimator.update(
            frame_index, participants, transformer)

        labels = []
        tracker_ids = final_detections.tracker_id
        if tracker_ids is not None:
            for tracker_id in tracker_ids:
                label = f"#{tracker_id}"
                if tracker_id in player_speeds_pps:
                    label += f" P: {player_speeds_pps[tracker_id]:.1f} px/s"
                    speed_mps = player_speeds_mps[tracker_id]
                    if speed_mps is not None:
                        label += f" | M: {speed_mps:.1f} m/s"
                labels.append(label)

        annotated_frame = ellipse_annotator.annotate(frame.copy(), final_detections)
        annotated_frame = label_annotator.annotate(
            annotated_frame, final_detections, labels=labels)
        annotated_frame = triangle_annotator.annotate(annotated_frame, ball_detections)
        if DRAW_TEAM_CONNECTIONS:
            for team_id in (0, 1):
                annotated_frame = draw_all_connections_on_frame(
                    annotated_frame, players[players.class_id == team_id],
                    color=palette.by_idx(team_id))
        sink.write_frame(annotated_frame)
        processed_frames += 1

if not processed_frames:
    raise RuntimeError("No video frames were decoded. Check the input video.")
print(f"Processing complete. Output video saved to {OUTPUT_VIDEO_PATH}")
if frames_without_calibration:
    print(f"Pitch calibration unavailable at {frames_without_calibration} speed samples; "
          "metric speed is omitted for those intervals.")